# End-to-End BI Data Validation Portfolio

Validazione completa tra layer source → staging → report usando dati reali CRM + SAP + Marketo.

Obiettivo: verificare accuratezza dati, riconciliazione revenue, rilevazione anomalie e regole di business.

In [25]:
import pandas as pd
import os
import sqlite3

print("Directory corrente:", os.getcwd())
print("File esiste?", os.path.exists('../data/crm_customers.csv'))

if os.path.exists('../data/crm_customers.csv'):
    df = pd.read_csv('../data/crm_customers.csv')
    print("Caricamento OK")
    display(df.head())
else:
    print("File NON trovato – controlla il percorso")
# Carica i CSV (adatta il percorso se necessario)
customers = pd.read_csv('../data/crm_customers.csv')
sales = pd.read_csv('../data/sap_sales.csv')
campaigns = pd.read_csv('../data/marketo_campaigns.csv')

# Crea DB SQLite in memory
conn = sqlite3.connect(':memory:')

customers.to_sql('crm_customers', conn, index=False)
sales.to_sql('sap_sales', conn, index=False)
campaigns.to_sql('marketo_campaigns', conn, index=False)

print("Dati caricati nei layer source")
display(customers.head(3))
display(sales.head(3))
display(campaigns.head(3))

Directory corrente: c:\Users\giorg\Desktop\bi-report-validation\notebooks
File esiste? True
Caricamento OK


,customer_id,customer_name,email,signup_date
0,1,Mario Rossi,mario.rossi@email.com,2024-01-15
1,2,Anna Bianchi,anna.bianchi@email.com,2024-02-10
2,3,Luca Verdi,luca.verdi@email.com,2024-03-05
3,4,Giulia Neri,giulia.neri@email.com,2024-01-20
4,5,Marco Blu,marco.blu@email.com,2024-04-01


Dati caricati nei layer source


,customer_id,customer_name,email,signup_date
0,1,Mario Rossi,mario.rossi@email.com,2024-01-15
1,2,Anna Bianchi,anna.bianchi@email.com,2024-02-10
2,3,Luca Verdi,luca.verdi@email.com,2024-03-05


,order_id,customer_id,amount,order_date
0,1001,1,150.50,2024-02-01
1,1002,2,299.99,2024-02-15
2,1003,3,89.00,2024-03-10


,campaign_id,customer_email,clicks,conversions
0,CAMP001,mario.rossi@email.com,45,5
1,CAMP002,anna.bianchi@email.com,120,12
2,CAMP003,luca.verdi@email.com,30,3


In [26]:
# Simulazione ETL: join CRM + SAP + Marketo + regola High Value
staging_query = """
SELECT 
    s.order_id,
    s.customer_id,
    c.customer_name,
    c.email,
    s.amount,
    s.order_date,
    m.clicks,
    m.conversions,
    CASE WHEN s.amount > 300 THEN 'High Value' ELSE 'Standard' END AS order_category
FROM sap_sales s
LEFT JOIN crm_customers c ON s.customer_id = c.customer_id
LEFT JOIN marketo_campaigns m ON c.email = m.customer_email
"""

staging = pd.read_sql_query(staging_query, conn)
staging.to_sql('staging', conn, index=False)

print("Layer staging creato con join e categoria")
display(staging.head(5))

Layer staging creato con join e categoria


,order_id,customer_id,customer_name,email,amount,order_date,clicks,conversions,order_category
0,1001,1,Mario Rossi,mario.rossi@email.com,150.50,2024-02-01,45.0,5,Standard
1,1001,1,Mario Rossi,mario.rossi@email.com,150.50,2024-02-01,45.0,5 -- duplicato campagna per stesso cliente,Standard
2,1002,2,Anna Bianchi,anna.bianchi@email.com,299.99,2024-02-15,120.0,12,Standard
3,1002,2,Anna Bianchi,anna.bianchi@email.com,299.99,2024-02-15,120.0,12 -- duplicato,Standard
4,1003,3,Luca Verdi,luca.verdi@email.com,89.00,2024-03-10,30.0,3,Standard


In [27]:
report_query = """
SELECT 
    customer_name,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(amount), 2) AS total_revenue,
    SUM(conversions) AS total_conversions
FROM staging
GROUP BY customer_name
HAVING total_revenue > 100
"""

report = pd.read_sql_query(report_query, conn)
print("Report finale aggregato")
display(report)

Report finale aggregato


,customer_name,total_orders,total_revenue,total_conversions
0,NaN,1,500.00,NaN
1,Anna Bianchi,6,5399.98,72.0
2,Francesco Nero,1,300.00,1.0
3,Giulia Neri,1,450.00,8.0
4,Luca Verdi,2,289.00,6.0
5,Mario Rossi,6,903.00,30.0


In [28]:
# 1. Riconciliazione revenue
source_revenue = pd.read_sql_query("SELECT ROUND(SUM(amount), 2) FROM sap_sales", conn).iloc[0,0]
staging_revenue = pd.read_sql_query("SELECT ROUND(SUM(amount), 2) FROM staging", conn).iloc[0,0]

print(f"Source revenue (SAP): {source_revenue}")
print(f"Staging revenue: {staging_revenue}")
print(f"MATCH: {source_revenue == staging_revenue}")

# 2. Clienti senza vendite
missing = pd.read_sql_query("""
SELECT c.customer_name, c.email
FROM crm_customers c
LEFT JOIN staging s ON c.customer_id = s.customer_id
WHERE s.order_id IS NULL
""", conn)
print("\nClienti senza vendite:")
display(missing)

# 3. Duplicati in source
duplicates = pd.read_sql_query("""
SELECT order_id, COUNT(*) as occurrences
FROM sap_sales
GROUP BY order_id
HAVING COUNT(*) > 1
""", conn)
print("\nDuplicati in source:")
display(duplicates)

Source revenue (SAP): 4640.49
Staging revenue: 7791.98
MATCH: False

Clienti senza vendite:


,customer_name,email
0,Marco Blu,marco.blu@email.com
1,Sara Rosa,sara.rosa@email.com
2,Antonio Verde,antonio.verde@email.com
3,Laura Arancione,laura.arancione@email.com



Duplicati in source:


,order_id,occurrences


## TC04 - Business rule High Value (>300)
Verifica che gli ordini >300 siano categorizzati correttamente come 'High Value'

In [29]:
# Query per TC04
high_value = pd.read_sql_query("""
SELECT 
    order_id,
    customer_id,
    customer_name,
    amount,
    order_date,
    order_category
FROM staging
WHERE amount > 300
""", conn)

print("Ordini High Value (>300) trovati")
display(high_value)

# Check manuale se la categoria è corretta
high_value_check = high_value['order_category'].eq('High Value').all()
print(f"Tutti categorizzati correttamente come High Value? {high_value_check}")

Ordini High Value (>300) trovati


,order_id,customer_id,customer_name,amount,order_date,order_category
0,1005,4,Giulia Neri,450.0,2024-03-20,High Value
1,1006,2,Anna Bianchi,1200.0,2024-04-05 -- High Value,High Value
2,1006,2,Anna Bianchi,1200.0,2024-04-05 -- High Value,High Value
3,1011,11,NaN,500.0,2024-07-01 -- cliente_id inesistente (non in...,High Value
4,1012,2,Anna Bianchi,1200.0,2024-04-05 -- duplicato di 1006,High Value
5,1012,2,Anna Bianchi,1200.0,2024-04-05 -- duplicato di 1006,High Value


Tutti categorizzati correttamente come High Value? True


## TC05 - Nessuna anomalia revenue per categoria (<100)
Verifica che non ci siano categorie con revenue totale troppo bassa (es. <100)

In [30]:
# Query per TC05
category_anomalies = pd.read_sql_query("""
SELECT 
    order_category,
    ROUND(SUM(amount), 2) AS category_revenue
FROM staging
GROUP BY order_category
HAVING category_revenue < 100
""", conn)

print("Categorie con revenue <100")
display(category_anomalies)

if category_anomalies.empty:
    print("0 anomalie rilevate → Test PASS")
else:
    print(f"{len(category_anomalies)} anomalie rilevate → Test FAIL")

Categorie con revenue <100


,order_category,category_revenue


0 anomalie rilevate → Test PASS


## Riassunto Test Cases & Esiti

In [31]:
print("Riassunto validazione:")
print("- Revenue match: ", "PASS" if source_revenue == staging_revenue else "FAIL")
print("- Clienti orfani trovati: ", len(missing))
print("- Duplicati in source: ", len(duplicates))
print("- Ordini High Value: ", len(high_value))
print("- Anomalie categoria <100: ", "0 (PASS)" if category_anomalies.empty else f"{len(category_anomalies)} (FAIL)")

Riassunto validazione:
- Revenue match:  FAIL
- Clienti orfani trovati:  4
- Duplicati in source:  0
- Ordini High Value:  6
- Anomalie categoria <100:  0 (PASS)
